Import

In [1]:
import sys
if 'utils' in sys.modules:
    del sys.modules['utils']
sys.path.append('../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, HeteroConv
from sklearn.metrics import roc_auc_score
import numpy as np
import os
from utils import EDGE_FEATURES

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

CONFIGS = {
    'A': ['comm'],
    'B': ['comm', 'context'],
    'C': ['comm', 'knowledge'],
    'D': ['comm', 'context', 'knowledge']
}

Device: mps


Definizione HeteroGNN

In [2]:
class HeteroGNN(nn.Module):
    def __init__(self, edge_types, hidden_channels=64):
        super().__init__()
        self.edge_types = edge_types
        
        self.conv1 = HeteroConv({
            ('node', etype, 'node'): GCNConv(-1, hidden_channels)
            for etype in edge_types
        }, aggr='sum')
        
        self.conv2 = HeteroConv({
            ('node', etype, 'node'): GCNConv(hidden_channels, hidden_channels)
            for etype in edge_types
        }, aggr='sum')
        
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        
        self.edge_classifier = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, 2)
        )
    
    def forward(self, data):
        x_dict = {'node': data['node'].x}
        edge_index_dict = {
            ('node', etype, 'node'): data['node', etype, 'node'].edge_index
            for etype in self.edge_types
        }
        
        out1 = self.conv1(x_dict, edge_index_dict)
        node_emb = torch.zeros(data['node'].num_nodes, 64, device=data['node'].x.device)
        for v in out1.values():
            node_emb = node_emb + v
        node_emb = self.bn1(F.relu(node_emb))
        
        x_dict2 = {'node': node_emb}
        out2 = self.conv2(x_dict2, edge_index_dict)
        node_emb2 = torch.zeros_like(node_emb)
        for v in out2.values():
            node_emb2 = node_emb2 + v
        node_emb2 = self.bn2(F.relu(node_emb2))
        
        return node_emb2
    
    def classify_edges(self, node_emb, edge_index):
        src = node_emb[edge_index[0]]
        dst = node_emb[edge_index[1]]
        edge_emb = torch.cat([src, dst], dim=1)
        return self.edge_classifier(edge_emb)

print("HeteroGNN definita.")

HeteroGNN definita.


Funzioni di Load e training

In [3]:
import os
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm  # <-- Import aggiunto per le progress bar

def load_graphs(config_name, split):
    split_dir = f'../outputs/graphs/CONFIG_{config_name}/{split}'
    graphs = []
    
    if not os.path.exists(split_dir):
        print(f"Warning: Path not found {split_dir}")
        return graphs

    for fname in sorted(os.listdir(split_dir)):
        if fname.endswith('.pt'):
            filepath = f'{split_dir}/{fname}'
            g = torch.load(filepath, weights_only=False)
            g.filepath = filepath
            graphs.append(g)
    return graphs

def train_one_epoch(model, graphs, optimizer, edge_types, device):
    model.train()
    total_loss = 0
    
    for data in graphs:
        data = data.to(device)
        optimizer.zero_grad()
        
        node_emb = model(data)
        losses = []
        
        for etype in edge_types:
            edge_index = data['node', etype, 'node'].edge_index
            edge_label = data['node', etype, 'node'].edge_label
            logits = model.classify_edges(node_emb, edge_index)
            
            ce_loss = F.cross_entropy(logits, edge_label, reduction='none')
            pt = torch.exp(-ce_loss)
            focal = ((1 - pt) ** 2 * ce_loss).mean()
            losses.append(focal)
        
        loss = sum(losses)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    return total_loss / len(graphs)

# Aggiunto il parametro 'desc' per personalizzare il testo della progress bar
def get_edge_labels(model, graphs, edge_types, device, threshold=0.5, desc="Labeling"):
    model.eval()
    results = []
    
    with torch.no_grad():
        # <-- Aggiunto tqdm qui per tracciare il processo di labeling sui grafi
        for data in tqdm(graphs, desc=desc, leave=False):
            data = data.to(device)
            node_emb = model(data)
            
            for etype in edge_types:
                edge_index = data['node', etype, 'node'].edge_index
                edge_label = data['node', etype, 'node'].edge_label
                logits = model.classify_edges(node_emb, edge_index)
                
                probs = F.softmax(logits, dim=1)[:, 1]
                preds = (probs > threshold).long()
                
                for i in range(edge_index.size(1)):
                    src = edge_index[0, i].item()
                    dst = edge_index[1, i].item()
                    
                    results.append({
                        'file_path': data.filepath,
                        'edge_type': etype,
                        'source_node': src,
                        'target_node': dst,
                        'true_label': edge_label[i].item(),
                        'predicted_label': preds[i].item(),
                        'probability': round(probs[i].item(), 4)
                    })
                    
    return results

print("Funzioni di load, training e labeling (con tqdm) definite.")

def evaluate(model, graphs, edge_types, device, threshold=0.5):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for data in graphs:
            data = data.to(device)
            node_emb = model(data)
            
            for etype in edge_types:
                edge_index = data['node', etype, 'node'].edge_index
                edge_label = data['node', etype, 'node'].edge_label
                logits = model.classify_edges(node_emb, edge_index)
                probs = F.softmax(logits, dim=1)[:, 1]
                preds = (probs > threshold).long()
                
                all_preds.append(preds.cpu())
                all_labels.append(edge_label.cpu())
                all_probs.append(probs.cpu())
    
    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    all_probs  = torch.cat(all_probs).numpy()
    
    tp = ((all_preds == 1) & (all_labels == 1)).sum().item()
    fp = ((all_preds == 1) & (all_labels == 0)).sum().item()
    tn = ((all_preds == 0) & (all_labels == 0)).sum().item()
    fn = ((all_preds == 0) & (all_labels == 1)).sum().item()
    
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    accuracy  = (tp + tn) / (tp + tn + fp + fn + 1e-8)
    auc       = roc_auc_score(all_labels.numpy(), all_probs)
    
    return {
        'precision': precision,
        'recall':    recall,
        'f1':        f1,
        'accuracy':  accuracy,
        'auc':       auc
    }

print("Funzioni definite.")

Funzioni di load, training e labeling (con tqdm) definite.
Funzioni definite.


Training con 5 epoch

In [4]:
import csv
from tqdm.auto import tqdm

EPOCHS = 5
HIDDEN = 64
all_edge_predictions = []

for config_name, edge_types in CONFIGS.items():
    print(f"\n{'='*40}")
    print(f"Training CONFIG_{config_name}")
    print('='*40)
    
    train_graphs = load_graphs(config_name, 'train')
    test_graphs  = load_graphs(config_name, 'test')
    
    train_graphs_filtered = [g for g in train_graphs
                   if g['node', 'comm', 'node'].edge_label.sum().item() > 0]
    print(f"  Grafi train con anomalie: {len(train_graphs_filtered)}")
    
    model = HeteroGNN(edge_types=edge_types, hidden_channels=HIDDEN).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)
    
    # <-- Progress bar per le epoche
    for epoch in tqdm(range(1, EPOCHS + 1), desc="Epochs"):
        if len(train_graphs_filtered) > 0:
            loss = train_one_epoch(model, train_graphs_filtered, optimizer, edge_types, device)
            # Usiamo tqdm.write invece di print per non "rompere" l'interfaccia della progress bar
            tqdm.write(f"  Epoch {epoch:02d} — Loss: {loss:.4f}")
    
    # -- ESTRAZIONE LABEL --
    # Passiamo il parametro desc per differenziare la progress bar visivamente
    train_labels = get_edge_labels(model, train_graphs, edge_types, device, desc="Labeling Train")
    test_labels = get_edge_labels(model, test_graphs, edge_types, device, desc="Labeling Test")
    
    for item in train_labels:
        item['config'] = config_name
        item['split'] = 'train'
        all_edge_predictions.append(item)
        
    for item in test_labels:
        item['config'] = config_name
        item['split'] = 'test'
        all_edge_predictions.append(item)

# -- EXPORT TO CSV --
output_csv = '../outputs/edge_labels_5ep.csv'
os.makedirs('../outputs', exist_ok=True) 

with open(output_csv, mode='w', newline='') as file:
    fieldnames = [
        'config', 'split', 'file_path', 'edge_type', 
        'source_node', 'target_node', 'true_label', 
        'predicted_label', 'probability'
    ]
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_edge_predictions)

print(f"\nDone. CSV con le label dei singoli archi salvato in: {output_csv}")


Training CONFIG_A
  Grafi train con anomalie: 16


Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

  Epoch 01 — Loss: 0.4007
  Epoch 02 — Loss: 0.2148
  Epoch 03 — Loss: 0.0530
  Epoch 04 — Loss: 0.0428
  Epoch 05 — Loss: 0.0358


Labeling Train:   0%|          | 0/39 [00:00<?, ?it/s]

Labeling Test:   0%|          | 0/18 [00:00<?, ?it/s]

...

Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

  Epoch 01 — Loss: 1.1563
  Epoch 02 — Loss: 0.8234
  Epoch 03 — Loss: 0.2992
  Epoch 04 — Loss: 0.1551
  Epoch 05 — Loss: 0.1331


Labeling Train:   0%|          | 0/39 [00:00<?, ?it/s]

Labeling Test:   0%|          | 0/18 [00:00<?, ?it/s]


Done. CSV con le label dei singoli archi salvato in: ../outputs/edge_labels_5ep.csv
